In [ ]:
%load_ext autoreload
%autoreload 2
import sys

ProjDIR = "/home/jw3514/Work/CellType_Psy/CellTypeBias_VIP/" # Change to your project directory
sys.path.insert(1, f'{ProjDIR}/src/')
from CellType_PSY import *
#import scanpy as sc
HGNC, ENSID2Entrez, GeneSymbol2Entrez, Entrez2Symbol = LoadGeneINFO()

try:
    os.chdir(f"{ProjDIR}/notebooks/")
    print(f"Current working directory: {os.getcwd()}")
except FileNotFoundError as e:
    print(f"Error: Could not change directory - {e}")
except Exception as e:
    print(f"Unexpected error: {e}")


import statsmodels.api as sm
from statsmodels.stats.multitest import fdrcorrection, multipletests

In [ ]:
import matplotlib.font_manager as fm
font_path = '/usr/share/fonts/truetype/msttcorefonts/Arial.ttf'
fm.fontManager.addfont(font_path)  # Only if you're adding a new font file
fm._load_fontmanager(try_read_cache=False)

# HumanCT IQ Phenotype


In [ ]:
HumanCT_Z2_HCT = pd.read_csv("/home/jw3514/Work/CellType_Psy/dat/HumanCTExpressionMats/HumanCT.TPM.0.1.Filt.Spec.clip.lowexp.cut1e4.csv", index_col=0)
HumanCT_Z2_HCT.columns = HumanCT_Z2_HCT.columns.astype(int)

In [ ]:
Mut_n_IQ = pd.read_csv("../dat/ASD_IQ_Mut.csv")
Mut_n_IQ.columns.values

In [ ]:
Spark_Denovo = pd.read_excel("../dat/41588_2022_1148_MOESM4_ESM.xlsx",
                           skiprows=2, sheet_name="Table S7")
Spark_Denovo = Spark_Denovo[Spark_Denovo[
    "pDenovoWEST_Meta"]!="."]
Spark_Denovo_ExomeWide = Spark_Denovo[Spark_Denovo[
    "pDenovoWEST_Meta"]<=1.3e-6]
Spark_Denovo_ExomeWide.shape

In [ ]:
top_Genes = Spark_Denovo.head(61)["HGNC"].values
Mut_n_IQ_conf = Mut_n_IQ[Mut_n_IQ["HGNC"].isin(top_Genes)]
Mut_n_IQ_conf.shape

In [ ]:
#### Gene Level
Genes = list(set(Mut_n_IQ_conf["Entrez"].values))
data = []
for g in Genes:
    tmp_df = Mut_n_IQ_conf[Mut_n_IQ_conf["Entrez"]==g]
    avg_IQ = tmp_df["IQ"].mean()
    row = [g, avg_IQ]
    data.append(row)
columns = ["Entrez", "IQ"]
Avg_Gene_IQ_DF = pd.DataFrame(data=data, columns=columns)

In [ ]:
HumanCT_res_df_MutL = Make_HumanCT_DF(Mut_n_IQ_conf, HumanCT_Z2_HCT, "../dat/Pheno_Bias_vs_IQ/HumanCT.spec.MutL.csv")
HumanCT_res_df_GeneL = Make_HumanCT_DF(Avg_Gene_IQ_DF, HumanCT_Z2_HCT, "../dat/Pheno_Bias_vs_IQ/HumanCT.spec.GeneL.csv")

In [ ]:
print(np.mean(HumanCT_res_df_GeneL[HumanCT_res_df_GeneL["Supercluster"]=="CGE interneuron"]["beta"]))
print(np.mean(HumanCT_res_df_GeneL[HumanCT_res_df_GeneL["Supercluster"]=="LAMP5-LHX6 and Chandelier"]["beta"]))
print(np.mean(HumanCT_res_df_GeneL[HumanCT_res_df_GeneL["Supercluster"]=="MGE interneuron"]["beta"]))

In [ ]:
HumanCT_res_df_GeneL.head(20)

In [ ]:
SuperClusterBias_BoxPlot_CorrIQ(HumanCT_res_df_GeneL, flip_axis=True, figsize=(6, 8), plot_metric="SpearmanR")

In [ ]:
SuperClusterBias_BoxPlot_CorrIQ(HumanCT_res_df_GeneL, flip_axis=True, figsize=(6, 8), plot_metric="beta")

In [ ]:
VIP_Anno = pd.read_csv("VIP_Anno.csv", index_col=0)

In [ ]:
# Get subset of rows from VIP_Anno that exist in X22q_Z2_Bias_PT
common_indices = HumanCT_res_df_GeneL.index.intersection(VIP_Anno.index)
HumanCT_res_df_GeneL_sub = HumanCT_res_df_GeneL.loc[common_indices].copy()
HumanCT_res_df_GeneL_sub["VIP"] = VIP_Anno.loc[common_indices, "VIP"]

In [ ]:
HumanCT_res_df_GeneL_sub_VIP_pos = HumanCT_res_df_GeneL_sub[HumanCT_res_df_GeneL_sub["VIP"] >= 1]
HumanCT_res_df_GeneL_sub_VIP_neg = HumanCT_res_df_GeneL_sub[HumanCT_res_df_GeneL_sub["VIP"] < 1]
# plot effect of VIP+ vs VIP-
data = [HumanCT_res_df_GeneL_sub_VIP_pos["beta"], HumanCT_res_df_GeneL_sub_VIP_neg["beta"]]
# Perform Mann-Whitney U test
stat, pval = scipy.stats.mannwhitneyu(HumanCT_res_df_GeneL_sub_VIP_pos["beta"], 
                                    HumanCT_res_df_GeneL_sub_VIP_neg["beta"])
# Create boxplot with individual points
bp = plt.boxplot(data, labels=["VIP+", "VIP-"])
# Add scatter points
for i, d in enumerate([HumanCT_res_df_GeneL_sub_VIP_pos["beta"], HumanCT_res_df_GeneL_sub_VIP_neg["beta"]]):
    x = np.random.normal(i+1, 0.04, size=len(d))
    plt.scatter(x, d, alpha=0.4, s=20)
plt.ylabel("Effect")
plt.title(f"p = {pval:.2e}")
plt.show()


In [ ]:
Effect = "SpearmanR"
HumanCT_res_df_GeneL_sub_VIP_pos = HumanCT_res_df_GeneL_sub[HumanCT_res_df_GeneL_sub["VIP"] >= 1]
HumanCT_res_df_GeneL_sub_VIP_neg = HumanCT_res_df_GeneL_sub[HumanCT_res_df_GeneL_sub["VIP"] < 1]
# plot effect of VIP+ vs VIP-
data = [HumanCT_res_df_GeneL_sub_VIP_pos[Effect], HumanCT_res_df_GeneL_sub_VIP_neg[Effect]]
# Perform Mann-Whitney U test
stat, pval = scipy.stats.mannwhitneyu(HumanCT_res_df_GeneL_sub_VIP_pos[Effect], 
                                    HumanCT_res_df_GeneL_sub_VIP_neg[Effect])
# Create boxplot with individual points
bp = plt.boxplot(data, labels=["VIP+", "VIP-"])
# Add scatter points
for i, d in enumerate([HumanCT_res_df_GeneL_sub_VIP_pos[Effect], HumanCT_res_df_GeneL_sub_VIP_neg[Effect]]):
    x = np.random.normal(i+1, 0.04, size=len(d))
    plt.scatter(x, d, alpha=0.4, s=20)
plt.ylabel("Effect")
plt.title(f"p = {pval:.2e}")
plt.show()

### Mut IQ permutation

In [ ]:
Perm_DIR = "/home/jw3514/Work/CellType_Psy/dat/Pheno_Bias_vs_IQ/IQ_Permuts/HumanCT_June13/ALL/"
Perm_DFs = []
for df in os.listdir(Perm_DIR):
    df = pd.read_csv(f"{Perm_DIR}/{df}", index_col=0)
    df.index = df.index.astype(int)
    Perm_DFs.append(df)
print(len(Perm_DFs))

In [ ]:
plot_null_distributions(HumanCT_res_df_GeneL, Perm_DFs, CT=295, plot=True)

In [ ]:
Supercluster = "CGE interneuron"
ClusterIdx = Anno[Anno["Supercluster"]==Supercluster].index.values

p_rho, p_beta = plot_null_suptercluster_distributions(ClusterIdx, HumanCT_res_df_GeneL, Perm_DFs, plot=True)

In [ ]:
#Supercluster = "CGE interneuron"
# Create DataFrame to store results
results_df = pd.DataFrame(columns=['Supercluster', 'p_rho', 'p_beta'])

for Supercluster in Neurons:
    ClusterIdx = Anno[Anno["Supercluster"]==Supercluster].index.values
    p_rho, p_beta = plot_null_suptercluster_distributions(ClusterIdx, HumanCT_res_df_GeneL, Perm_DFs, plot=False)
    
    # Add results to DataFrame
    results_df = results_df.append({
        'Supercluster': Supercluster,
        "mean_PBS": HumanCT_res_df_GeneL.loc[ClusterIdx, "beta"].mean(),
        'p_rho': p_rho,
        'p_beta': p_beta
    }, ignore_index=True)

In [ ]:
# Add FDR corrected p-values
results_df['p_rho_FDR'] = multipletests(results_df['p_rho'], method='fdr_bh')[1]
results_df['p_beta_FDR'] = multipletests(results_df['p_beta'], method='fdr_bh')[1]

results_df.sort_values("p_beta")

In [ ]:
for i, row in HumanCT_res_df_GeneL.iterrows():
    ct = int(row["CT"])
    p_rho, p_beta = plot_null_distributions(HumanCT_res_df_GeneL, Perm_DFs, CT=ct, plot=False)
    HumanCT_res_df_GeneL.loc[i, "p_rho_perm"] = p_rho
    HumanCT_res_df_GeneL.loc[i, "p_beta_perm"] = p_beta
#HumanCT_res_df_GeneL.sort_values("p_beta_FDR")

In [ ]:
HumanCT_res_df_GeneL_neuron = HumanCT_res_df_GeneL[HumanCT_res_df_GeneL["Supercluster"].isin(Neurons)]

In [ ]:
HumanCT_res_df_GeneL = HumanCT_res_df_GeneL.sort_values("p_beta_perm")
HumanCT_res_df_GeneL_neuron = HumanCT_res_df_GeneL_neuron.sort_values("p_beta_perm")
print(HumanCT_res_df_GeneL.shape, HumanCT_res_df_GeneL_neuron.shape)


In [ ]:
def safe_neglog10(x):
    val = -np.log10(x)
    if np.isinf(val):
        return 4.0 # 4.0 is the max value for p-values, change according to number of permutations (10,000)
    return val

HumanCT_res_df_GeneL["p_beta_perm_Log"] = HumanCT_res_df_GeneL["p_beta_perm"].apply(safe_neglog10)

In [ ]:
HumanCT_res_df_GeneL = HumanCT_res_df_GeneL.sort_values("p_beta_perm")
HumanCT_res_df_GeneL.head(5)

In [ ]:
#umanCT_res_df_GeneL
SuperClusterBias_BoxPlot_CorrIQ(HumanCT_res_df_GeneL, flip_axis=False, figsize=(6, 8), plot_metric="p_beta_perm_Log", xlabel="PBS -log10(P)")

In [ ]:
common_indices = HumanCT_res_df_GeneL.index.intersection(VIP_Anno.index)
HumanCT_res_df_GeneL_sub = HumanCT_res_df_GeneL.loc[common_indices].copy()
HumanCT_res_df_GeneL_sub["VIP"] = VIP_Anno.loc[common_indices, "VIP"]

In [ ]:
HumanCT_res_df_GeneL_sub_VIP_pos = HumanCT_res_df_GeneL_sub[HumanCT_res_df_GeneL_sub["VIP"] >= 1]
HumanCT_res_df_GeneL_sub_VIP_neg = HumanCT_res_df_GeneL_sub[HumanCT_res_df_GeneL_sub["VIP"] < 1]
# plot effect of VIP+ vs VIP-
#EFFECT = "beta"
EFFECT = "p_beta_perm_Log"
data = [HumanCT_res_df_GeneL_sub_VIP_pos[EFFECT], HumanCT_res_df_GeneL_sub_VIP_neg[EFFECT]]
# Perform Mann-Whitney U test
stat, pval = scipy.stats.mannwhitneyu(HumanCT_res_df_GeneL_sub_VIP_pos[EFFECT], 
                                    HumanCT_res_df_GeneL_sub_VIP_neg[EFFECT])
# Create boxplot with individual points
bp = plt.boxplot(data, labels=["VIP+", "VIP-"])
# Add scatter points
for i, d in enumerate([HumanCT_res_df_GeneL_sub_VIP_pos[EFFECT], HumanCT_res_df_GeneL_sub_VIP_neg[EFFECT]]):
    x = np.random.normal(i+1, 0.04, size=len(d))
    plt.scatter(x, d, alpha=0.4, s=20)
plt.ylabel(EFFECT)
plt.title(f"p = {pval:.2e}")
plt.show()


In [ ]:
HumanCT_res_df_GeneL['p_rho_perm_FDR'] = multipletests(HumanCT_res_df_GeneL['p_rho_perm'], method='fdr_bh')[1]
HumanCT_res_df_GeneL['p_beta_perm_FDR'] = multipletests(HumanCT_res_df_GeneL['p_beta_perm'], method='fdr_bh')[1]

HumanCT_res_df_GeneL_neuron['p_rho_perm_FDR'] = multipletests(HumanCT_res_df_GeneL_neuron['p_rho_perm'], method='fdr_bh')[1]
HumanCT_res_df_GeneL_neuron['p_beta_perm_FDR'] = multipletests(HumanCT_res_df_GeneL_neuron['p_beta_perm'], method='fdr_bh')[1]

In [ ]:
HumanCT_res_df_GeneL_neuron.head(10)

In [ ]:
HumanCT_res_df_GeneL = HumanCT_res_df_GeneL.sort_values("beta")

In [ ]:
HumanCT_res_df_GeneL.head(10)

In [ ]:
HumanCT_res_df_GeneL.to_csv("../dat/Pheno_Bias_vs_IQ/HumanCT.GeneL.cluster.June10.csv")
results_df = results_df.sort_values("p_beta")
results_df.to_csv("../dat/Pheno_Bias_vs_IQ/HumanCT.GeneL.Supercluster.June10.csv")

In [ ]:
results_df

#### Test VIP+ and VIP-

In [ ]:
CGE_VIP_Pos_list = np.loadtxt("../dat/Other/CGE_VIP_Pos.txt", dtype=int)
CGE_VIP_Neg_list = np.loadtxt("../dat/Other/CGE_VIP_Neg.txt", dtype=int)

In [ ]:
VIP_pos_meanPBS = HumanCT_res_df_GeneL[HumanCT_res_df_GeneL.index.isin(CGE_VIP_Pos_list)]["beta"].mean()
VIP_neg_meanPBS = HumanCT_res_df_GeneL[HumanCT_res_df_GeneL.index.isin(CGE_VIP_Neg_list)]["beta"].mean()
DIFF_obs = VIP_pos_meanPBS - VIP_neg_meanPBS
print(VIP_pos_meanPBS, VIP_neg_meanPBS, VIP_pos_meanPBS - VIP_neg_meanPBS)

In [ ]:
null_diffs = []
for i in range(10000):
    tmpDF = Perm_DFs[i]
    tmp_VIP_pos_meanPBS = tmpDF[tmpDF.index.isin(CGE_VIP_Pos_list)]["beta"].mean()
    tmp_VIP_neg_meanPBS = tmpDF[tmpDF.index.isin(CGE_VIP_Neg_list)]["beta"].mean()
    null_diffs.append(tmp_VIP_pos_meanPBS - tmp_VIP_neg_meanPBS)
null_diffs = np.array(null_diffs)

In [ ]:
# Plot observed difference vs null distribution
plt.figure(figsize=(8,6))
plt.hist(null_diffs, bins=30, density=True, alpha=0.5, label='Null distribution')
plt.axvline(DIFF_obs, color='red', linestyle='dashed', label='Observed difference')
plt.xlabel('VIP+ vs VIP- mean bias difference')
plt.ylabel('Density')
plt.legend()

#compute 1-sided p-value
pval = np.mean(null_diffs <= DIFF_obs)
print(f"Observed difference: {DIFF_obs:.3f}")
print(f"P-value: {pval:.3e}")

### LGD vs Dmis

In [ ]:
Mut_n_IQ_conf_LGD = Mut_n_IQ_conf[Mut_n_IQ_conf["GeneEff"]!="missense"]
Mut_n_IQ_conf_Dmis = Mut_n_IQ_conf[Mut_n_IQ_conf["GeneEff"]=="missense"]
print(Mut_n_IQ_conf.shape, Mut_n_IQ_conf_LGD.shape, Mut_n_IQ_conf_Dmis.shape)

In [ ]:
# Calculate gene level for Mut_n_IQ_conf
Genes_LGD = list(set(Mut_n_IQ_conf_LGD["Entrez"].values))
data_LGD = []
for g in Genes_LGD:
    tmp_df = Mut_n_IQ_conf_LGD[Mut_n_IQ_conf_LGD["Entrez"]==g]
    avg_IQ = tmp_df["IQ"].mean()
    row = [g, avg_IQ]
    data_LGD.append(row)
columns = ["Entrez", "IQ"]
Avg_Gene_IQ_DF_LGD = pd.DataFrame(data=data_LGD, columns=columns)

# Calculate gene level for Mut_n_IQ_Dmis
Genes_Dmis = list(set(Mut_n_IQ_conf_Dmis["Entrez"].values))
data_Dmis = []
for g in Genes_Dmis:
    tmp_df = Mut_n_IQ_conf_Dmis[Mut_n_IQ_conf_Dmis["Entrez"]==g]
    avg_IQ = tmp_df["IQ"].mean()
    row = [g, avg_IQ]
    data_Dmis.append(row)
Avg_Gene_IQ_DF_Dmis = pd.DataFrame(data=data_Dmis, columns=columns)



In [ ]:
HumanCT_res_df_GeneL_LGD = Make_HumanCT_DF(Avg_Gene_IQ_DF_LGD, HumanCT_Z2_HCT, "../dat/Pheno_Bias_vs_IQ/HumanCT.GeneL.LGD.csv")
HumanCT_res_df_GeneL_Dmis = Make_HumanCT_DF(Avg_Gene_IQ_DF_Dmis, HumanCT_Z2_HCT, "../dat/Pheno_Bias_vs_IQ/HumanCT.GeneL.Dmis.csv")

In [ ]:
SuperClusterBias_BoxPlot_CorrIQ(HumanCT_res_df_GeneL_LGD, flip_axis=True, figsize=(6, 8))

In [ ]:
SuperClusterBias_BoxPlot_CorrIQ(HumanCT_res_df_GeneL_Dmis, flip_axis=True, figsize=(6, 8))

In [ ]:
Perm_DIR_LGD = "/home/jw3514/Work/CellType_Psy/dat/Pheno_Bias_vs_IQ/IQ_Permuts/HumanCT_Feb25/LGD/"
Perm_DFs_LGD = []

max_count = 100000

i = 0
for df in os.listdir(Perm_DIR_LGD):
    df = pd.read_csv(f"{Perm_DIR_LGD}/{df}", index_col=0)
    df.index = df.index.astype(int)
    Perm_DFs_LGD.append(df)
    i += 1
    if i > max_count:
        break
print(len(Perm_DFs_LGD))

Perm_DIR_Dmis = "/home/jw3514/Work/CellType_Psy/dat/Pheno_Bias_vs_IQ/IQ_Permuts/HumanCT_Feb25/Dmis/"
Perm_DFs_Dmis = []

max_count = 100000

i = 0
for df in os.listdir(Perm_DIR_Dmis):
    df = pd.read_csv(f"{Perm_DIR_Dmis}/{df}", index_col=0)
    df.index = df.index.astype(int)
    Perm_DFs_Dmis.append(df)
    i += 1
    if i > max_count:
        break
print(len(Perm_DFs_Dmis))

In [ ]:
HumanCT_res_df_GeneL_LGD

In [ ]:
results_df_LGD = pd.DataFrame(columns=['Supercluster', 'p_rho', 'p_beta'])

for Supercluster in Neurons:
    ClusterIdx = Anno[Anno["Supercluster"]==Supercluster].index.values
    p_rho_LGD, p_beta_LGD = plot_null_suptercluster_distributions(ClusterIdx, HumanCT_res_df_GeneL_LGD, Perm_DFs_LGD, plot=False)
    
    # Add results to DataFrame
    results_df_LGD = results_df_LGD.append({
        'Supercluster': Supercluster,
        'p_rho': p_rho_LGD,
        'p_beta': p_beta_LGD
    }, ignore_index=True)

In [ ]:
results_df_Dmis = pd.DataFrame(columns=['Supercluster', 'p_rho', 'p_beta'])

for Supercluster in Neurons:
    ClusterIdx = Anno[Anno["Supercluster"]==Supercluster].index.values
    p_rho_Dmis, p_beta_Dmis = plot_null_suptercluster_distributions(ClusterIdx, HumanCT_res_df_GeneL_Dmis, Perm_DFs_Dmis, plot=False)
    
    # Add results to DataFrame
    results_df_Dmis = results_df_Dmis.append({
        'Supercluster': Supercluster,
        'p_rho': p_rho_Dmis,
        'p_beta': p_beta_Dmis
    }, ignore_index=True)

In [ ]:
results_df_LGD

In [ ]:
results_df_Dmis

# UKBB